In [1]:
import numpy as np
import sys
import matplotlib.pyplot as plt
import torch
import torcheval
import torch_geometric

sys.path.append("../")

In [2]:
DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_pixel_spacetime.npy"
BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_mppc_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_mppc_spacetime.npy"
SIGNAL_ONLY_PIXEL_FILE = f"{DATA_DIR}/sig_only_pixel_spacetime.npy"
SIGNAL_ONLY_MPPC_FILE = f"{DATA_DIR}/sig_only_mppc_spacetime.npy"


bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_MPPC_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
sig_mppc_spacetime = np.load(SIGNAL_MPPC_FILE)


def get_labelled_points(
    bg_pixel_spacetime, bg_mppc_spacetime, sig_pixel_spacetime, sig_mppc_spacetime
):
    X_pixel = np.concatenate((bg_pixel_spacetime, sig_pixel_spacetime), axis=0)
    X_mppc = np.concatenate((bg_mppc_spacetime, sig_mppc_spacetime), axis=0)
    y = np.concatenate(
        (np.zeros(bg_pixel_spacetime.shape[0]), np.ones(sig_pixel_spacetime.shape[0])),
        axis=0,
    )

    # Shuffle the data
    indices = np.arange(len(y))
    np.random.shuffle(indices)
    X_pixel = X_pixel[indices]
    X_mppc = X_mppc[indices]
    y = y[indices]

    pixel_label = np.zeros(X_pixel.shape[:-1] + (1,))
    pixel_label[(X_pixel[:, :, :] == -1).all(axis=-1)] = -1
    X_pixel_labelled = np.concatenate((X_pixel, pixel_label), axis=-1)
    mppc_label = np.ones(X_mppc.shape[:-1] + (1,))
    mppc_label[(X_mppc[:, :, :] == -1).all(axis=-1)] = -1
    X_mppc_labelled = np.concatenate((X_mppc, mppc_label), axis=-1)

    X = np.concatenate((X_pixel_labelled, X_mppc_labelled), axis=1)
    return X, y

X, y = get_labelled_points(
    bg_pixel_spacetime, bg_mppc_spacetime, sig_pixel_spacetime, sig_mppc_spacetime
)

In [3]:
import torch
from torch_geometric.data import Data
from torch_geometric.nn import knn_graph


def set_to_graph(features, label):
    # remove padded nodes (all zeros)
    mask = ~(features == -1).all(axis=1)
    x = torch.tensor(features[mask], dtype=torch.float)

    n = x.size(0)
    if n == 0:
        return None  # skip empty graphs

    y = torch.tensor([label], dtype=torch.float)
    return Data(x=x, y=y)


graphs = [set_to_graph(s, l) for s, l in zip(X[:100000], y[:100000])]
graphs = [g for g in graphs if g is not None]  # filter out None
del X, y  # free memory

In [4]:
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

test_graphs, val_graphs = train_test_split(graphs, test_size=0.1, random_state=42)


loader = DataLoader(test_graphs, batch_size=32, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=32, shuffle=False)

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, EdgeConv, DynamicEdgeConv


class GNN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super(GNN, self).__init__()
        self.num_classes = num_classes
        self.conv1 = DynamicEdgeConv(
            nn=nn.Sequential(
                nn.Linear(2 * in_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            ),
            k=5,
        )
        self.conv2 = DynamicEdgeConv(
            nn=nn.Sequential(
                nn.Linear(2 * hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            ),
            k=10,
        )
        self.conv3 = DynamicEdgeConv(
            nn=nn.Sequential(
                nn.Linear(2 * hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            ),
            k=5,
        )
        self.conv4 = DynamicEdgeConv(
            nn=nn.Sequential(
                nn.Linear(2 * hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            ),
            k=5,
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, data):
        x = data.x
        x = self.conv1(x, batch=data.batch)  # Edge convolution
        x = self.conv2(x, batch=data.batch)  # Edge convolution
        x = self.conv3(x, batch=data.batch)  # Dynamic edge convolution
        x = self.conv4(x, batch=data.batch)  # Edge convolution
        x = torch_geometric.nn.global_mean_pool(x, data.batch)  # Global pooling
        x = self.fc(x).squeeze()
        return torch.sigmoid(x) if self.num_classes == 1 else F.log_softmax(x, dim=1)

In [6]:
model = GNN(in_dim=5, hidden_dim=10, num_classes=1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
from torcheval.metrics.aggregation.auc import AUC
print(f"Model has {sum(p.numel() for p in model.parameters() if p.requires_grad)} trainable parameters.")
auc_metric = AUC()


for epoch in range(20):
    auc_metric.reset()  # Reset metric at the start of each epoch
    for batch_data in loader:
        optimizer.zero_grad()
        out = model(batch_data)
        loss = criterion(out, batch_data.y)
        loss.backward()
        optimizer.step()
        break
    val_loss = []
    for batch_data in val_loader:
        with torch.no_grad():
            val_loss.append(criterion(model(batch_data), batch_data.y))
            out = model(batch_data)
            auc_metric.update(out, batch_data.y)
        break
    val_loss = torch.mean(torch.tensor(val_loss))
    print(f"Epoch {epoch}, Loss {loss.item():.4f}, Val Loss {val_loss.item():.4f} - AUC {auc_metric.compute().item():.4f}")

Model has 1191 trainable parameters.
Epoch 0, Loss 51.9889, Val Loss 48.5215 - AUC 0.0004
Epoch 1, Loss 41.5904, Val Loss 48.5215 - AUC 0.0004
Epoch 2, Loss 48.5214, Val Loss 48.5215 - AUC 0.0004
Epoch 3, Loss 45.0559, Val Loss 48.5215 - AUC 0.0004
Epoch 4, Loss 51.9861, Val Loss 48.5214 - AUC 0.0004
Epoch 5, Loss 72.7811, Val Loss 48.5214 - AUC 0.0004
Epoch 6, Loss 34.6576, Val Loss 48.5214 - AUC 0.0003
Epoch 7, Loss 48.5205, Val Loss 48.5214 - AUC 0.0003
Epoch 8, Loss 51.9875, Val Loss 48.5214 - AUC 0.0003
Epoch 9, Loss 58.9157, Val Loss 48.5214 - AUC 0.0003
Epoch 10, Loss 41.5898, Val Loss 48.5214 - AUC 0.0003
Epoch 11, Loss 51.9866, Val Loss 48.5214 - AUC 0.0003
Epoch 12, Loss 58.9179, Val Loss 48.5213 - AUC 0.0003
Epoch 13, Loss 55.4523, Val Loss 48.5213 - AUC 0.0003
Epoch 14, Loss 38.1243, Val Loss 48.5213 - AUC 0.0003
Epoch 15, Loss 48.5214, Val Loss 48.5213 - AUC 0.0003
Epoch 16, Loss 51.9862, Val Loss 48.5213 - AUC 0.0003
Epoch 17, Loss 58.9175, Val Loss 48.5213 - AUC 0.0003
E

In [7]:
import torch
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

tensor([1.], device='mps:0')


In [1]:
import torch, torch.nn.functional as F
from torch import nn, optim
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# dataset
data = Planetoid(root="data", name="Cora")[0].to(device)

# model
class GCN(nn.Module):
    def __init__(self, in_ch, hid, out_ch, dropout=0.5):
        super().__init__()
        self.c1 = GCNConv(in_ch, hid)
        self.c2 = GCNConv(hid, out_ch)
        self.dp = nn.Dropout(dropout)
    def forward(self, x, edge_index):
        x = self.c1(x, edge_index); x = F.relu(x); x = self.dp(x)
        x = self.c2(x, edge_index)
        return x

model = GCN(data.num_features, 64, int(data.y.max())+1).to(device)
opt = optim.AdamW(model.parameters(), lr=3e-3, weight_decay=5e-4)

# Optional: mixed precision on MPS (bf16/float16). bfloat16 needs macOS 14+.
use_amp = (device.type == "mps")
amp_dtype = torch.bfloat16  # try torch.float16 if you like

for epoch in range(200):
    model.train()
    opt.zero_grad(set_to_none=True)
    if use_amp:
        with torch.autocast("mps", dtype=amp_dtype):
            out = model(data.x, data.edge_index)
            loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    else:
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])

    loss.backward()
    opt.step()

    model.eval()
    with torch.inference_mode():
        logits = model(data.x, data.edge_index)
        pred = logits.argmax(dim=-1)
        acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean()
    if (epoch+1) % 20 == 0:
        print(f"epoch {epoch+1:3d} | loss {loss.item():.4f} | test acc {acc.item():.3f}")


Processing...
Done!


epoch  20 | loss 0.4225 | test acc 0.800
epoch  40 | loss 0.0674 | test acc 0.792
epoch  60 | loss 0.0236 | test acc 0.788
epoch  80 | loss 0.0162 | test acc 0.784
epoch 100 | loss 0.0093 | test acc 0.785
epoch 120 | loss 0.0086 | test acc 0.786
epoch 140 | loss 0.0069 | test acc 0.782
epoch 160 | loss 0.0037 | test acc 0.784
epoch 180 | loss 0.0043 | test acc 0.785
epoch 200 | loss 0.0034 | test acc 0.784
